# 四脚ロボットの歩行方策を Genesis で学習する（Google Colab）

『つくりながら学ぶ！リアルタイムOS自作入門』第13章の四脚の歩行方策を、物理シミュレータ
**Genesis** で学習するノートブックです。学習した方策は MuJoCo でも歩かせて比べます
（**sim2sim**）。手順と結果はサポートページの「Genesis で学習して MuJoCo で歩かせる」に
まとめてあります。

やること:

1. 環境を作る（genesis-world + rsl_rl + MuJoCo）
2. **配布済みの方策を Genesis で歩かせる** — 環境が正しく組めたかの確認
3. 割り当てられた GPU での**学習速度を実測**し、所要時間を見積もる
4. 学習する（チェックポイントは Google Drive に置いて、切れても続けられるように）
5. 学習した方策を Genesis と MuJoCo の両方で歩かせて比べ、**歩くところを動画で見る**
6. 実機用の `.h` と、ブラウザ版の `policy_genesis.json` に書き出す

**先に「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選んでください。**

| | |
|---|---|
| 元のコード | [code/13_quadruped/rl/genesis](https://github.com/iory/build-your-own-arduino-rtos/tree/main/code/13_quadruped/rl/genesis) |
| サポートページ | https://iory.github.io/build-your-own-arduino-rtos/ |

## 0. GPU の確認

ここで GPU が出てこない場合は、**ランタイム → ランタイムのタイプを変更**で
GPU を選び直してから、このセルをもう一度実行してください。

In [ ]:
import subprocess, sys

print(sys.version)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU:", gpu.stdout.strip())
else:
    raise SystemExit("GPU が割り当てられていません。"
                     "ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください")

## 1. コードを取ってくる

サポートページのリポジトリから第13章のコード一式（ロボットの MJCF、学習コード、
配布済みの方策）を取ってきます。`--sparse` で必要なディレクトリだけを落とします。

In [ ]:
REF = "main"   # 取ってくるブランチ

import os
os.environ["REF"] = REF
%cd /content
!rm -rf /content/build-your-own-arduino-rtos
!git clone --depth 1 --branch "$REF" --filter=blob:none --sparse \
    https://github.com/iory/build-your-own-arduino-rtos.git
!git -C build-your-own-arduino-rtos sparse-checkout set code/13_quadruped
GENESIS_DIR = "/content/build-your-own-arduino-rtos/code/13_quadruped/rl/genesis"
assert os.path.exists(f"{GENESIS_DIR}/gs_env.py"), (
    f"ブランチ {REF!r} に rl/genesis がありません。REF を確かめて、このセルをもう一度実行してください")
!ls "{GENESIS_DIR}"

## 2. 学習環境を入れる（数分）

版は `rl/genesis/pyproject.toml` と同じものに固定します。torch は Colab に最初から
入っているものを使います。

| | |
|---|---|
| genesis-world | 1.4.4 |
| rsl-rl-lib | 5.0.1（mjlab 版・CPU 版と同じ。チェックポイントの形式がそろう） |
| mujoco | 3.7.0（sim2sim の MuJoCo 側。ブラウザ版と同じ物理） |
| tensorboard | 版は固定しない（学習のログを書く） |

pip が「既存のパッケージと衝突する」という警告を出すことがありますが、下のセルの
最後で import が通れば問題ありません。**「ランタイムを再起動してください」と
言われたら再起動して、セクション 3 から再実行**してください。

In [ ]:
!pip install -q genesis-world==1.4.4 rsl-rl-lib==5.0.1 mujoco==3.7.0 tensorboard 2>&1 | tail -5
!python -c "import genesis, rsl_rl, mujoco, torch, tensorboard; print('genesis', genesis.__version__, '| mujoco', mujoco.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. 場所と学習の設定を環境変数で渡す

`ARDUINO_QUAD_HOME_HEIGHT`（立つ高さ）と `ARDUINO_QUAD_GAIT_PERIOD`（歩容の周期）は、
配布している方策と同じ値にします。書き出しのときに、学習したときの値と食い違うと
止まるようにしてあるので、学習から書き出しまで同じ値のままにしてください。

**ランタイムが再起動したら、このセルからやり直してください。** ディスクの中身
（クローンとインストール）は残っていますが、環境変数は消えています。

In [ ]:
import os

ROOT = "/content/build-your-own-arduino-rtos/code/13_quadruped"
GENESIS = f"{ROOT}/rl/genesis"
os.environ["ROOT"] = ROOT
os.environ["GENESIS"] = GENESIS
os.environ["ARDUINO_QUAD_HOME_HEIGHT"] = "0.11"
os.environ["ARDUINO_QUAD_GAIT_PERIOD"] = "0.32"
os.environ["PYTHONUNBUFFERED"] = "1"

assert os.path.exists(f"{GENESIS}/gs_env.py"), "rl/genesis が見つかりません（セクション 1 を先に）"
%cd $GENESIS

## 4. 配布済みの方策を Genesis で歩かせる

学習を始める前に、**環境が正しく組めているか**を確かめます。歩かせるのは
MuJoCo で学習して実機に載せている方策です。サポートページの実測では、指令 0.12 m/s で
Genesis でも 0.117 m/s で歩き、1 台も倒れません。ここで同じくらいの数字が出れば、
Genesis・方策・観測の組み立ての 3 つがそろって正常です。

初回は Genesis が GPU のカーネルをコンパイルするので、数分かかります。

In [ ]:
!python sim2sim.py --npz ../../arduino_quad_policy.npz --sim genesis \
    --commands "0.12,0" --envs 32 --seconds 12 2>&1 | grep -v "dubious center of mass"

## 5. この GPU での学習速度を測る

Colab が割り当てる GPU は日によって（T4 / L4 / A100）変わり、学習にかかる時間も
変わります。**30 iteration だけ回して実測**し、そこから所要時間を見積もります。
最初の数 iteration はカーネルのコンパイルを含むので、見積もりからは外します。

比較用に、RTX 4090（4096 環境）での実測は **0.72〜0.79 秒/iteration** です。
このぶんのログは捨てます（次のセクションで本番の学習を最初から始めます）。

In [ ]:
NUM_ENVS = 4096

os.environ["NUM_ENVS"] = str(NUM_ENVS)
!python train.py --num-envs "$NUM_ENVS" --iterations 30 --run-name colab_bench 2>&1 \
    | grep --line-buffered -v "dubious center of mass" | tee /content/bench.log | grep -E "Learning iteration|Iteration time|Traceback|Error"

In [ ]:
import re

text = open("/content/bench.log", encoding="utf-8", errors="replace").read()
times = [float(v) for v in re.findall(r"Iteration time:\s+([\d.]+)s", text)]
fps = [int(v) for v in re.findall(r"Steps per second:\s+(\d+)", text)]
assert times, "ログを読めませんでした（上のセルの出力を確認してください）"

warm = 10 if len(times) > 15 else 0
per_iter = sum(times[warm:]) / len(times[warm:])
print(f"1 iteration あたり {per_iter:.2f} 秒（{NUM_ENVS} 環境、{len(times) - warm} iteration の平均）")
print(f"シミュレーション速度 {sum(fps[warm:]) / len(fps[warm:]):,.0f} steps/s\n")
print("  iteration     見積もり時間")
for it in (500, 1000, 1500, 2300):
    print(f"  {it:>9,}   {per_iter * it / 3600:>6.1f} 時間")
print("\n配布している方策は 2300 iteration（4096 環境）まで回したものです。")

## 6. チェックポイントを Google Drive に置く

Colab のセッションは切れます（無料枠では数時間、放置すると 90 分程度）。
切れるとローカルディスクは消えるので、**学習ログとチェックポイントを Drive に
逃がして**おきます。Drive を使いたくない場合はこのセルを飛ばせます。その場合、
セッションが切れた時点で学習結果は失われます。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

os.environ["LOGS"] = "/content/drive/MyDrive/arduino_quad_genesis/logs"
!mkdir -p "$LOGS"
!rm -rf "$GENESIS/logs"
!ln -sfn "$LOGS" "$GENESIS/logs"
!ls -l "$GENESIS/logs/"

## 7. 学習する

- `--num-envs` は並列に回す機体の数です。配布している方策は 4096 で学習しました
- `--iterations` は方策を更新する回数です。配布している方策は 2300 です
- `--seed` で乱数の種を変えると、別の方策になります（配布しているのは seed 1）

チェックポイントは 100 iteration ごとに `logs/rsl_rl/arduino_quad_velocity/<日時>_colab/`
へ保存されます。**ブラウザのタブを閉じないでください**（閉じるとセッションが
切れます）。途中で止めても、保存済みのチェックポイントからセクション 8 で再開できます。

手元の RTX 4090 で回したときは、iteration 200 くらいまではすぐ倒れ、400 で
ほとんど倒れなくなり、900 で指令 0.09 m/s に対して 0.085 m/s で歩きました。

In [ ]:
ITERATIONS = 2300
SEED = 1

os.environ["ITERATIONS"] = str(ITERATIONS)
os.environ["SEED"] = str(SEED)
!python train.py --num-envs "$NUM_ENVS" --iterations "$ITERATIONS" --seed "$SEED" --run-name colab 2>&1 \
    | grep --line-buffered -v "dubious center of mass" | grep --line-buffered -E "Learning iteration|Mean episode length|Mean reward|ETA|iterations in|Traceback|Error"

## 8. 切れた続きから再開する

セッションが切れたら、**セクション 0 → 1 → 2 → 3 → 5 の最初のセル（`NUM_ENVS` を決める
だけでよい）→ 6 を実行し直して**から、このセルを実行します。いちばん新しい
チェックポイントを読み、`ITERATIONS` 回ぶん**追加で**学習します（2300 まで回したいなら、
残りの回数を入れてください）。

In [ ]:
import glob

ckpts = sorted(glob.glob(f"{GENESIS}/logs/rsl_rl/arduino_quad_velocity/*/model_*.pt"),
               key=lambda p: (os.path.dirname(p), int(p.split("model_")[-1][:-3])))
assert ckpts, "チェックポイントがありません"
os.environ["RESUME"] = ckpts[-1]
print("続きから:", ckpts[-1])

ITERATIONS = 1000   # 追加で回す回数
os.environ["ITERATIONS"] = str(ITERATIONS)
!python train.py --num-envs "$NUM_ENVS" --iterations "$ITERATIONS" --seed "$SEED" --run-name colab \
    --resume "$RESUME" 2>&1 \
    | grep --line-buffered -v "dubious center of mass" | grep --line-buffered -E "Learning iteration|Mean episode length|Mean reward|ETA|iterations in|Traceback|Error"

## 9. Genesis と MuJoCo で歩かせて比べる（sim2sim）

いちばん新しいチェックポイントを、Genesis と MuJoCo の両方で同じ指令で歩かせます。
MuJoCo 側はブラウザ版と同じ物理です。

| 列 | 意味 |
|---|---|
| `vx` / `wz` | 後半 10 秒の前進 [m/s] と旋回 [rad/s] |
| `roll` / `pitch` | 胴体の傾きの RMS [°] |
| `falls` | 倒れた回数 |
| `duty` | 4 本の足それぞれが接地している時間の割合（最小〜最大） |
| `diag` | 対角 2 本が同時に接地／離地している割合（きれいなトロットほど 1 に近い） |
| `lift` / `slip` | 足先の持ち上げの高さ [mm] と、接地中の足先の滑り [mm/s] |

In [ ]:
import glob

ckpts = sorted(glob.glob(f"{GENESIS}/logs/rsl_rl/arduino_quad_velocity/*/model_*.pt"),
               key=lambda p: (os.path.dirname(p), int(p.split("model_")[-1][:-3])))
assert ckpts, "チェックポイントがありません（セクション 7 を先に）"
os.environ["CKPT"] = ckpts[-1]
print("使うチェックポイント:", ckpts[-1])

In [ ]:
!python sim2sim.py --checkpoint "$CKPT" --envs 64 --seconds 20 2>&1 \
    | grep -E "^ *(genesis|mujoco|sim) |Traceback|Error"

## 10. 歩くところを動画で見る

1 台だけ歩かせて、MuJoCo の描画機能で mp4 に書き出し、このノートブックの中で
再生します。指令は 4 秒ごとに、前進 0.06 m/s → 前進 0.12 m/s → 左旋回 0.3 rad/s →
前進しながら右旋回、と切り替わります（`--commands` と `--seconds` で変えられます）。

- **左: MuJoCo で歩かせたもの**（ブラウザ版と同じ物理。sim2sim の本命）
- **右: Genesis で歩かせたもの**（学習したシミュレータ）

どちらも絵を描くのは MuJoCo で、ロボットの見た目（メッシュ）は同じです。Colab には
画面が無いので、GPU の EGL で描画します（MuJoCo 公式の Colab チュートリアルと
同じ設定: https://github.com/google-deepmind/mujoco/blob/main/python/tutorial.ipynb ）。

学習が足りずセクション 9 で倒れていた方策だと、動画でも倒れては起き直す
（リセットされる）様子が映ります。

In [ ]:
import json
import subprocess

# Colab の GPU で EGL 描画を使うための設定（MuJoCo 公式チュートリアルと同じ）
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        json.dump({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}, f)
os.environ["MUJOCO_GL"] = "egl"

assert os.path.exists(f"{GENESIS}/render.py"), (
    f"{GENESIS}/render.py がありません。セクション 1 で取ってきたブランチ {os.environ.get('REF')!r} が古いので、"
    "下の「つまずいたら」の手順で render.py だけを取ってきてください")
for sim in ("mujoco", "genesis"):
    r = subprocess.run(["python", "render.py", "--checkpoint", os.environ["CKPT"], "--sim", sim,
                        "--out", f"/content/walk_{sim}.mp4"], cwd=GENESIS, capture_output=True, text=True)
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "dubious center of mass" not in l]
    if r.returncode != 0:
        print("\n".join(lines[-30:]))
        raise SystemExit(f"render.py（{sim}）が失敗しました。上の出力を見てください")
    print([l for l in lines if l.startswith("wrote")][-1])

In [ ]:
from base64 import b64encode
from IPython.display import HTML


def video_tag(path, label):
    data = b64encode(open(path, "rb").read()).decode()
    return (f'<figure style="margin:0"><video width="100%" controls autoplay loop muted playsinline '
            f'src="data:video/mp4;base64,{data}"></video><figcaption>{label}</figcaption></figure>')


HTML('<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">'
     + video_tag("/content/walk_mujoco.mp4", "MuJoCo（ブラウザ版と同じ物理）")
     + video_tag("/content/walk_genesis.mp4", "Genesis（学習したシミュレータ）")
     + "</div>")

## 11. 書き出す

実機用のヘッダ（`arduino_quad_policy.h` / `.npz` / `.json`）と、ブラウザ版の歩行
シミュレーションが読む `policy_genesis.json` を作ります。重みだけでなく、観測の並び・
履歴の順序・正規化の統計・home 角・action のスケールまで書き込み、numpy で
再実装した順伝播と PyTorch の出力を突き合わせて検証します。

`policy_genesis.json` は、リポジトリの `docs/assembly/quadruped/walk/` に置き換えると
ブラウザ版の「RL 方策（Genesis で学習）」になります。**実機に書き込むときは、
最初は必ず機体を吊るしてから**動かしてください。

In [ ]:
!python ../../host/export_quad_policy.py "$CKPT" /content/exported --backend genesis 2>&1 | tail -4
!python ../../host/make_walk_bundle.py /content/exported /content/exported/policy_genesis.json
!ls -l /content/exported

In [ ]:
from google.colab import files

for name in ("arduino_quad_policy.h", "arduino_quad_policy.npz", "arduino_quad_policy.json",
             "policy_genesis.json"):
    files.download(f"/content/exported/{name}")

## つまずいたら

| 症状 | 原因と対処 |
|---|---|
| `GPU が割り当てられていません` | ランタイム → ランタイムのタイプを変更 → T4 GPU |
| `rl/genesis が見つかりません` | セクション 1 を実行し直す |
| `ModuleNotFoundError: genesis` | セクション 2 のインストールが途中で切れています。もう一度 |
| `render.py がありません` | セクション 1 で取ってきたブランチに render.py が入っていません。クローンを消さずに、新しいセルで `!git -C /content/build-your-own-arduino-rtos fetch --depth 1 origin feat/genesis-colab` と `!git -C /content/build-your-own-arduino-rtos checkout FETCH_HEAD -- code/13_quadruped/rl/genesis/render.py` を実行してからセクション 10 をもう一度（セクション 1 をやり直すと、Drive に置いていない学習結果も消えます） |
| `CUDA out of memory` | セクション 5 の `NUM_ENVS` を 2048 に下げて、セクション 5 からやり直す |
| `export env does not match the training run` | セクション 3 の `ARDUINO_QUAD_HOME_HEIGHT` / `ARDUINO_QUAD_GAIT_PERIOD` が学習したときと違います |
| セッションが切れた | セクション 0→1→2→3→5（最初のセル）→6 をやり直して、セクション 8 で再開 |

質問は [GitHub Issues](https://github.com/iory/build-your-own-arduino-rtos/issues) へどうぞ。